# Статистический анализ точности системы лазерного наведения с БПЛА

Анализ посвящён исследованию точности системы лазерного наведения, установленной на квадрокоптере со стабилизированным подвесом.

Основная метрика качества — **радиальная ошибка** $R = \sqrt{r_x^2 + r_y^2}$ (мм), измеренная как горизонтальное отклонение лазерного пятна от реперной точки на поверхности.

### Характеристика выборки

| Параметр | Значение |
|---|---|
| Период проведения испытаний | апрель — октябрь 2025 г. |
| Число лётных дней | 59 |
| Контрольные точки | 5 (P1–P5) |
| Высоты зависания | $H \in \{3,\; 5,\; 10,\; 15,\; 20\}$ м |
| Повторов на каждую комбинацию (точка × высота) | 3 |
| Общее число измерений | $59 \times 5 \times 5 \times 3 = 4425$ |

Для каждого измерения фиксировались: время, координаты точки, высота $H$, номер повтора, локальная скорость ветра $V$ (м/с), порывы ветра, направление ветра, облачность (%), число спутников GNSS, а также компоненты отклонения $r_x$, $r_y$ и радиальная ошибка $R$.

In [94]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

BASE_DIR = Path(".").resolve()
OUTPUT_DIR = BASE_DIR / "output"
JSON_PATH = BASE_DIR / "balanced_selection" / "selected_days_2024-2025.json"

H_ORDER = [3, 5, 10, 15, 20]
H_CAT_ORDER = ["3 м", "5 м", "10 м", "15 м", "20 м"]
ALPHA = 0.05

COL_MAP = {
    "Время": "time", "Точка": "point", "H (м)": "H", "Полёт": "flight",
    "V (м/с)": "V", "Порывы, откр. источник (м/с)": "gusts",
    "Направление ветра": "wind_dir", "Облачность (%)": "cloud",
    "Спутники": "sat", "r_x (мм)": "r_x", "r_y (мм)": "r_y", "R (мм)": "R"
}


def load_measurements(input_dir: Path) -> pd.DataFrame:
    frames = []
    for fpath in sorted(input_dir.glob("*.xlsx")):
        tmp = pd.read_excel(fpath, header=12, engine="openpyxl")
        tmp.rename(columns=COL_MAP, inplace=True)
        tmp["date"] = fpath.stem
        frames.append(tmp)
    df = pd.concat(frames, ignore_index=True)
    for c in ("H", "V", "cloud", "sat", "R", "r_x", "r_y", "gusts", "wind_dir"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df.dropna(subset=["H", "V", "R"], inplace=True)
    return df


weather = pd.DataFrame(json.loads(JSON_PATH.read_text(encoding="utf-8")))
weather["date"] = pd.to_datetime(weather["date"])

df = load_measurements(OUTPUT_DIR)
df["H_cat"] = df["H"].astype(int).astype(str) + " м"
df["V_bin"] = pd.cut(
    df["V"],
    bins=[0, 2.2, 2.9, 3.6, 4.4, 100],
    labels=["0.8–2.2", "2.2–2.9", "2.9–3.6", "3.6–4.4", "4.4–7.8"],
)

print(f"Загружено: {len(df)} измерений за {df['date'].nunique()} дней")
print(f"Погодный JSON: {len(weather)} дней")
print(f"Высоты: {sorted(df['H'].unique())} м")
print(f"Точки: {', '.join(sorted(df['point'].unique()))}")
print(f"Повторов на высоту: {df['flight'].nunique()}")

ALL_FIGS: list[tuple[str, go.Figure]] = []


Загружено: 4425 измерений за 59 дней
Погодный JSON: 59 дней
Высоты: [np.int64(3), np.int64(5), np.int64(10), np.int64(15), np.int64(20)] м
Точки: P1, P2, P3, P4, P5
Повторов на высоту: 3


### Обзор выборки и условий эксперимента

Перед анализом ошибки наведения необходимо убедиться, что выборка лётных дней охватывает достаточно разнообразные метеорологические условия и не содержит систематического смещения по какому-либо фактору. Ниже приведено распределение дней по типу погоды и гистограммы ключевых метеопараметров.

In [95]:
weather_counts = weather["weather_code_text"].value_counts().reset_index()
weather_counts.columns = ["Погода", "Дней"]

fig = px.bar(
    weather_counts, x="Погода", y="Дней",
    color="Погода", text="Дней",
    title="Рис. 1. Распределение лётных дней по типу погоды",
)
fig.update_layout(showlegend=False, xaxis_title="", yaxis_title="Количество дней")
fig.update_traces(textposition="outside")
ALL_FIGS.append(("fig_01_weather_types", fig))
fig.show()

In [96]:
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    "Температура (°C)", "Средний ветер (м/с)",
    "Максимальные порывы (м/с)", "Облачность (%)"
])

for i, (col, name) in enumerate([
    ("temperature_2m_mean", "Температура"),
    ("wind_speed_10m_mean", "Ветер"),
    ("wind_gusts_10m_max", "Порывы"),
    ("cloud_cover_mean", "Облачность"),
]):
    r, c = divmod(i, 2)
    fig.add_trace(
        go.Histogram(x=weather[col], nbinsx=15, name=name, showlegend=False),
        row=r + 1, col=c + 1
    )

fig.update_layout(height=500, title_text="Рис. 2. Распределение метеопараметров по дням выборки")
ALL_FIGS.append(("fig_02_meteo_histograms", fig))
fig.show()

In [97]:
fig = px.histogram(
    df, x="R", nbins=80, marginal="box",
    title="Рис. 3. Общее распределение радиальной ошибки R",
    labels={"R": "R (мм)"},
    color_discrete_sequence=["steelblue"],
)
fig.update_layout(yaxis_title="Количество измерений", height=400)
ALL_FIGS.append(("fig_03_R_distribution", fig))
fig.show()

Выборка охватывает разнообразные погодные условия — от ясных дней со слабым ветром до пасмурных с порывами. Благодаря сбалансированной выборке представлены все ветровые режимы от штиля до сильного ветра.

### Описательная статистика

Для первичной характеристики выборки вычислим основные числовые показатели радиальной ошибки $R$ в каждой высотной группе: объём подвыборки $N$, среднее $\bar{R}$, стандартное отклонение $\sigma$, минимум, максимум и 95-й процентиль $R$

In [98]:
desc = (
    df.groupby("H", sort=True)["R"]
    .agg(
        N="count",
        Среднее="mean",
        Ст_откл="std",
        Минимум="min",
        Максимум="max",
        P95=lambda x: x.quantile(0.95)
    )
    .round(1)
)
desc.index.name = "H (м)"
desc.columns = ["N", "Среднее, мм", "σ, мм", "Мин, мм", "Макс, мм", "R₉₅, мм"]

fig = go.Figure(data=[go.Table(
    columnwidth=[50, 50, 80, 70, 60, 70, 70],
    header=dict(
        values=["<b>H (м)</b>"] + [f"<b>{c}</b>" for c in desc.columns],
        fill_color="#34495e", font=dict(color="white", size=13),
        align="center", height=32,
    ),
    cells=dict(
        values=[desc.index.tolist()] + [desc[c].tolist() for c in desc.columns],
        fill_color=[["#ecf0f1", "#ffffff"] * 3],
        font=dict(size=12), align="center", height=28,
    )
)])
fig.update_layout(
    title="Таблица 1. Описательная статистика радиальной ошибки R по высотам",
    width=820, height=260, margin=dict(l=10, r=10, t=50, b=10),
    font=dict(family="Arial"),
)
ALL_FIGS.append(("fig_desc_table", fig))
fig.show()

n_per_h = len(df) // df["H"].nunique()
n_days = df["date"].nunique()
print(f"{n_per_h} измерений на высоту: {n_days} дней × 5 точек × 3 повтора = {n_per_h}")

print("\nОписательная статистика непрерывных ковариат:")
for col, unit in [("V", "м/с"), ("gusts", "м/с"), ("cloud", "%"), ("sat", "шт.")]:
    vals = df[col].dropna()
    print(f"  {col}: среднее = {vals.mean():.1f} {unit}, σ = {vals.std():.1f}, "
          f"диапазон [{vals.min():.1f} – {vals.max():.1f}]")

885 измерений на высоту: 59 дней × 5 точек × 3 повтора = 885

Описательная статистика непрерывных ковариат:
  V: среднее = 3.4 м/с, σ = 1.4, диапазон [0.7 – 8.7]
  gusts: среднее = 9.9 м/с, σ = 3.1, диапазон [3.1 – 19.2]
  cloud: среднее = 60.1 %, σ = 33.9, диапазон [0.0 – 100.0]
  sat: среднее = 22.4 шт., σ = 3.7, диапазон [16.0 – 31.0]


По описательным характеристикам видны две закономерности:

1. **Монотонный рост среднего** $\bar{R}(H)$ с увеличением высоты. При наклоне подвеса на угол $\delta\varphi$ горизонтальное смещение составляет $\Delta R \approx H \cdot \delta\varphi$, то есть линейно возрастает с высотой.

2. **Стандартное отклонение растёт вместе со средним** — чем выше $H$, тем больше не только $\bar{R}$, но и разброс $\sigma$. Ошибка ведёт себя пропорционально: на большой высоте растёт и среднее, и непредсказуемость.

Это подтверждает доминирующее влияние фактора высоты $H$.


### Распределение ошибки по высотам

In [99]:
fig = px.box(
    df, x="H", y="R", color="H_cat",
    title="Рис. 4. Распределение радиальной ошибки R по высоте полёта",
    labels={"H": "Высота H (м)", "R": "Радиальная ошибка R (мм)", "H_cat": "Высота"},
    category_orders={"H_cat": H_CAT_ORDER},
)
fig.update_layout(showlegend=False, height=500, width=850)
ALL_FIGS.append(("fig_04_boxplot_H", fig))
fig.show()

In [100]:
fig = px.violin(
    df, x="H", y="R", color="H_cat", box=True, points=False,
    title="Рис. 5. Форма распределения R по высотам (скрипичная диаграмма)",
    labels={"H": "Высота H (м)", "R": "Радиальная ошибка R (мм)", "H_cat": "Высота"},
    category_orders={"H_cat": H_CAT_ORDER},
)
fig.update_traces(width=0.85, spanmode="hard")
fig.update_layout(showlegend=False, height=600, width=900, violingap=0.15)
ALL_FIGS.append(("fig_05_violin_H", fig))
fig.show()

In [101]:
stats_by_h = df.groupby("H")["R"].agg(["mean", "std", "count"]).reset_index()
stats_by_h["se"] = stats_by_h["std"] / np.sqrt(stats_by_h["count"])
stats_by_h["ci95"] = stats_by_h["se"] * 1.96

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=stats_by_h["H"], y=stats_by_h["mean"],
    mode="lines+markers", name="Среднее R",
    line=dict(width=3), marker=dict(size=10),
))
fig.add_trace(go.Scatter(
    x=list(stats_by_h["H"]) + list(stats_by_h["H"][::-1]),
    y=list(stats_by_h["mean"] + stats_by_h["ci95"])
      + list((stats_by_h["mean"] - stats_by_h["ci95"])[::-1]),
    fill="toself", fillcolor="rgba(99,110,250,0.15)",
    line=dict(width=0), name="95% дов. интервал", showlegend=True,
))
fig.update_layout(
    title="Рис. 6. Средняя ошибка R по высоте (с 95% доверительным интервалом)",
    xaxis_title="Высота H (м)", yaxis_title="R (мм)", height=400,
)
ALL_FIGS.append(("fig_06_mean_R_ci", fig))
fig.show()

На рис. 4–5 отчётливо видно, что с ростом высоты $H$:

- **медиана** и **среднее** $R$ систематически смещаются вверх;
- **межквартильный размах** расширяется — разброс ошибок увеличивается;
- **правый хвост** распределения удлиняется, что указывает на появление больших выбросов при неблагоприятных условиях.

Физическая интерпретация: при угловой нестабильности подвеса $\delta\varphi$ горизонтальное отклонение составляет

$$\Delta R \approx H \cdot \tan(\delta\varphi) \approx H \cdot \delta\varphi,$$

на практике прирост $\bar{R}(H)$ **ускоряется** к участку 15–20 м (суперпозиция геометрического рычага и ветра), а не следует одной прямой пропорциональности $H$. На рис. 6 подтверждается монотонный рост $\bar{R}(H)$ с неперекрывающимися доверительными интервалами на всех высотах.

### Влияние рельефа на высотах 15–20 м

Испытания проводились на территории карьера. На высотах до 10–15 м квадрокоптер находится внутри карьерной выемки, где рельеф частично экранирует ветровое воздействие. При подъёме на 15–20 м аппарат выходит над бровкой карьера и попадает в открытый ветровой поток.

Ожидаемый эффект: на высотах 15–20 м к геометрическому рычагу $H \cdot \delta\varphi$ добавляется ветровая составляющая, ранее ослабленная рельефом. Это должно проявляться в **ускорении роста ошибки** и в **увеличении разницы** между измерениями при слабом и сильном ветре.

In [102]:
v_q25 = df["V"].quantile(0.25)
v_q75 = df["V"].quantile(0.75)

low_wind = df[df["V"] <= v_q25].groupby("H")["R"].median().reset_index()
low_wind.columns = ["H", "R_median"]
low_wind["Режим"] = f"Слабый ветер (V ≤ {v_q25:.1f} м/с)"

high_wind = df[df["V"] >= v_q75].groupby("H")["R"].median().reset_index()
high_wind.columns = ["H", "R_median"]
high_wind["Режим"] = f"Сильный ветер (V ≥ {v_q75:.1f} м/с)"

plateau_df = pd.concat([low_wind, high_wind])

fig = px.line(
    plateau_df, x="H", y="R_median", color="Режим",
    markers=True,
    title="Рис. 7. Медианная ошибка R по высоте: слабый и сильный ветер",
    labels={"H": "Высота H (м)", "R_median": "Медианная R (мм)"},
)
fig.update_traces(line=dict(width=3), marker=dict(size=10))
fig.update_layout(height=450)
ALL_FIGS.append(("fig_07_plateau_wind", fig))
fig.show()

In [103]:
heights = sorted(df["H"].unique())
median_by_h = df.groupby("H")["R"].median()

deltas = []
for i in range(1, len(heights)):
    h_prev, h_curr = heights[i - 1], heights[i]
    delta = median_by_h[h_curr] - median_by_h[h_prev]
    deltas.append({"Интервал высот": f"{int(h_prev)}→{int(h_curr)} м", "ΔR": delta})

delta_df = pd.DataFrame(deltas)

fig = px.bar(
    delta_df, x="Интервал высот", y="ΔR", text="ΔR",
    title="Рис. 8. Прирост медианной ошибки между соседними высотами",
    labels={"Интервал высот": "Интервал высот (переход)", "ΔR": "ΔR (мм)"},
    color_discrete_sequence=["coral"],
)
fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig.update_layout(height=400, yaxis_title="Прирост ΔR (мм)", xaxis_title="Интервал высот (переход)")
ALL_FIGS.append(("fig_08_delta_R", fig))
fig.show()

Рис. 7–8 подтверждают ожидаемый эффект рельефа:

1. **Ускорение роста ошибки.** Прирост медианной ошибки $\Delta \tilde{R}$ при переходе 15→20 м составляет наибольшее значение среди всех пар соседних высот (рис. 8). Темп роста ошибки не замедляется, а увеличивается — на этих высотах к геометрическому рычагу добавляется ветровое воздействие, ранее ослабленное стенками карьера.

2. **Расхождение ветровых кривых.** На высотах 15–20 м разрыв между линиями «слабый ветер» и «сильный ветер» (рис. 7) заметно увеличивается. При выходе БПЛА из аэродинамической «тени» карьера система становится существенно более чувствительной к внешнему ветру.

Физическая интерпретация: на малых высотах стенки карьера экранируют ветер, и ошибка определяется преимущественно высотным рычагом $\Delta R \approx H \cdot \delta\varphi$. Выше бровки к нему добавляется ветровая составляющая, что приводит к выраженному взаимодействию факторов $H$ и $V$.

### Влияние ветра

Скорость ветра $V$ — второй по значимости фактор. Ветер раскачивает платформу БПЛА и подвес, увеличивая амплитуду колебаний $\delta\varphi$ и, как следствие, ошибку $R$. При порывах эффект усиливается нелинейно.

In [104]:
fig = px.scatter(
    df, x="V", y="R", color="H_cat",
    title="Рис. 9. Зависимость R от скорости ветра V (цвет — высота)",
    labels={"V": "Скорость ветра V (м/с)", "R": "R (мм)", "H_cat": "Высота"},
    category_orders={"H_cat": H_CAT_ORDER},
    opacity=0.35, trendline="ols",
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(height=500)
ALL_FIGS.append(("fig_09_R_vs_V", fig))
fig.show()

In [105]:
wind_bins = ["0.8–2.2", "2.2–2.9", "2.9–3.6", "3.6–4.4", "4.4–7.8"]

median_pivot = df.pivot_table(
    values="R", index="H", columns="V_bin", aggfunc="median", observed=False
).reindex(columns=wind_bins)
mean_pivot = df.pivot_table(
    values="R", index="H", columns="V_bin", aggfunc="mean", observed=False
).reindex(columns=wind_bins)

fig = make_subplots(
    rows=2,
    cols=1,
    vertical_spacing=0.18,
)

fig.add_trace(
    go.Heatmap(
        z=median_pivot.to_numpy(),
        x=median_pivot.columns.tolist(),
        y=median_pivot.index.tolist(),
        colorscale="YlOrRd",
        colorbar=dict(title="R (мм)", x=1.02, y=0.82, len=0.34),
        text=np.round(median_pivot.to_numpy(), 0),
        texttemplate="%{text:.0f}",
        hovertemplate="Высота H: %{y} м<br>Режим ветра: %{x}<br>Медиана R: %{z:.1f} мм<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Heatmap(
        z=mean_pivot.to_numpy(),
        x=mean_pivot.columns.tolist(),
        y=mean_pivot.index.tolist(),
        colorscale="Blues",
        colorbar=dict(title="R (мм)", x=1.02, y=0.18, len=0.34),
        text=np.round(mean_pivot.to_numpy(), 0),
        texttemplate="%{text:.0f}",
        hovertemplate="Высота H: %{y} м<br>Режим ветра: %{x}<br>Среднее R: %{z:.1f} мм<extra></extra>",
    ),
    row=2,
    col=1,
)

fig.update_layout(
    title="Рис. 10. Ошибка R (мм): медиана и среднее по высоте и режиму ветра",
    height=800,
)
fig.update_xaxes(side="top", row=1, col=1)
fig.update_xaxes(side="top", row=2, col=1)
fig.update_yaxes(
    title_text="Высота H (м)",
    autorange="reversed",
    tickmode="array",
    tickvals=[3, 5, 10, 15, 20],
    ticktext=["3", "5", "10", "15", "20"],
    row=1,
    col=1,
)
fig.update_yaxes(
    title_text="Высота H (м)",
    autorange="reversed",
    tickmode="array",
    tickvals=[3, 5, 10, 15, 20],
    ticktext=["3", "5", "10", "15", "20"],
    row=2,
    col=1,
)
ALL_FIGS.append(("fig_10_heatmap_HV", fig))
fig.show()

На рис. 9 наклон линий регрессии положителен для всех высотных групп: ошибка $R$ возрастает при увеличении $V$. При этом **наклон увеличивается с высотой** — подтверждается наличие взаимодействия факторов $H \times V$.

Тепловая карта (рис. 10) даёт ту же картину в обобщённом виде: правый нижний угол (сильный ветер + большая высота) соответствует наибольшей медианной ошибке, а левый верхний — наименьшей. Градиент по вертикали (высота) выражен сильнее, чем по горизонтали (ветер), что согласуется с иерархией $H > V$.

### Вторичные факторы: направление ветра, спутники, облачность

Помимо скорости ветра и высоты, фиксировались направление ветра (азимут), число видимых спутников GNSS и облачность. Проверим их вклад в ошибку $R$.

In [106]:
sample_rxy = df.sample(n=min(2000, len(df)), random_state=42)

fig = px.scatter(
    sample_rxy, x="r_x", y="r_y", color="wind_dir",
    color_continuous_scale="HSV",
    title="Рис. 11. Компоненты ошибки r_x и r_y (цвет — направление ветра)",
    labels={"r_x": "r_x (мм)", "r_y": "r_y (мм)", "wind_dir": "Направл. (°)"},
    opacity=0.5,
)
fig.update_traces(marker=dict(size=4))

r_max = max(sample_rxy["r_x"].abs().max(), sample_rxy["r_y"].abs().max()) * 1.05
radii = np.linspace(r_max * 0.25, r_max, 4)
theta_circle = np.linspace(0, 2 * np.pi, 361)

for r in radii:
    fig.add_trace(go.Scatter(
        x=r * np.cos(theta_circle), y=r * np.sin(theta_circle),
        mode="lines", line=dict(color="gray", width=0.5, dash="dot"),
        showlegend=False, hoverinfo="skip",
    ))

compass = {"С": 90, "СВ": 45, "В": 0, "ЮВ": 315, "Ю": 270, "ЮЗ": 225, "З": 180, "СЗ": 135}
for label, deg in compass.items():
    rad = np.radians(deg)
    fig.add_trace(go.Scatter(
        x=[0, r_max * np.cos(rad)], y=[0, r_max * np.sin(rad)],
        mode="lines", line=dict(color="lightgray", width=0.5),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_annotation(
        x=r_max * 1.08 * np.cos(rad), y=r_max * 1.08 * np.sin(rad),
        text=label, showarrow=False, font=dict(size=10, color="gray"),
    )

fig.update_layout(
    height=620, width=650,
    xaxis=dict(scaleanchor="y", range=[-r_max * 1.15, r_max * 1.15]),
    yaxis=dict(range=[-r_max * 1.15, r_max * 1.15]),
)
ALL_FIGS.append(("fig_11_rxy_winddir", fig))
fig.show()

На рис. 11 распределение $(r_x, r_y)$ в ядре близко к **радиально симметричному**; цвет (азимут ветра) **перемешан** по квадрантам, выраженного «вытягивания» вдоль направления ветра не видно. Возможен лишь **слабый** совместный дрейф; о сильной направленной связи азимута ветра и вектора ошибки по этому графику судить нельзя.

In [107]:
df["sat_group"] = pd.qcut(df["sat"], 4, labels=False, duplicates="drop")
sat_labels = (
    df.groupby("sat_group")["sat"]
    .agg(["min", "max"])
    .apply(lambda r: f"{int(r['min'])}–{int(r['max'])}", axis=1)
)
df["sat_label"] = df["sat_group"].map(sat_labels)
sat_order = sorted(sat_labels.values, key=lambda s: int(s.split("–")[0]))

fig = px.box(
    df, x="sat_label", y="R",
    title="Рис. 12. Ошибка R по квартилям числа спутников",
    labels={"sat_label": "Спутники (квартили)", "R": "R (мм)"},
    color_discrete_sequence=["mediumpurple"],
    category_orders={"sat_label": sat_order},
)
fig.update_layout(height=400)
ALL_FIGS.append(("fig_12_sat_boxplot", fig))
fig.show()

Рис. 12 показывает, что число спутников GNSS имеет заметное влияние на ошибку: медиана $R$ снижается при увеличении числа видимых спутников. Большее число спутников обеспечивает более точное позиционирование автопилота и, как следствие, более стабильное удержание позиции.

In [108]:
factors = ["H", "V", "sat", "cloud"]
factor_names = {"H": "Высота", "V": "Ветер", "sat": "Спутники", "cloud": "Обл."}
corrs = []
for f in factors:
    rho, p = stats.spearmanr(df[f], df["R"])
    corrs.append({
        "Фактор": factor_names[f],
        "ρ Спирмена": rho,
        "p-значение": p,
        "|ρ|": abs(rho)
    })

corr_df = pd.DataFrame(corrs)

fig = px.bar(
    corr_df, x="Фактор", y="ρ Спирмена", text="ρ Спирмена",
    color="|ρ|", color_continuous_scale="Blues",
    title="Рис. 13. Ранговая корреляция Спирмена каждого фактора с R",
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(height=400, coloraxis_showscale=False, yaxis_title="ρ Спирмена")
ALL_FIGS.append(("fig_13_spearman", fig))
fig.show()


Сводная ранговая корреляция Спирмена (рис. 13) подтверждает иерархию: $|\rho_H| \approx |\rho_V| > |\rho_{\text{sat}}| > |\rho_{\text{cloud}}|$. Высота и ветер лидируют с близкими по модулю корреляциями, спутники занимают третье место (отрицательная связь: больше спутников — меньше ошибка). Облачность имеет слабую положительную корреляцию с $R$, однако она обусловлена в первую очередь мультиколлинеарностью: пасмурные дни сопровождаются более сильным ветром, и именно ветер, а не облачность, является причиной роста ошибки.


### Проверка предпосылок дисперсионного анализа

Перед проведением дисперсионного анализа необходимо проверить выполнение его основных предпосылок:

- **Нормальность распределения** остатков внутри каждой группы (критерий Шапиро–Уилка).
- **Однородность дисперсий** (гомоскедастичность) между группами (критерий Ливеня).

#### Критерий Шапиро–Уилка

Критерий Шапиро–Уилка проверяет нулевую гипотезу $H_0$: выборка извлечена из нормально распределённой генеральной совокупности. Статистика $W \in [0, 1]$; значения $W$, близкие к единице, свидетельствуют о хорошем согласии с нормальным законом.

In [109]:
H_ORDER = sorted(df["H"].unique())
groups_by_H = [df.loc[df["H"] == h, "R"].values for h in H_ORDER]

shapiro_rows = []
for h, grp in zip(H_ORDER, groups_by_H):
    sample = (
        grp if len(grp) <= 5000
        else np.random.default_rng(42).choice(grp, 5000, replace=False)
    )
    W, p = stats.shapiro(sample)
    shapiro_rows.append([
        int(h), len(grp), round(W, 4), f"{p:.2e}",
        "Норм." if p > ALPHA else f"W={W:.4f}"
    ])

fig = go.Figure(data=[go.Table(
    columnwidth=[60, 60, 80, 100, 120],
    header=dict(
        values=["<b>H (м)</b>", "<b>N</b>", "<b>W</b>", "<b>p-значение</b>", "<b>Заключение</b>"],
        fill_color="#34495e", font=dict(color="white", size=13),
        align="center", height=32,
    ),
    cells=dict(
        values=list(zip(*shapiro_rows)),
        fill_color=[["#ecf0f1", "#ffffff"] * 3],
        font=dict(size=12), align="center", height=28,
    )
)])
fig.update_layout(
    title="Таблица 2. Результаты критерия Шапиро–Уилка по высотным группам",
    width=700, height=260, margin=dict(l=10, r=10, t=50, b=10),
    font=dict(family="Arial"),
)
ALL_FIGS.append(("fig_shapiro_table", fig))
fig.show()

При объёме подвыборки $N = 900$ критерий Шапиро–Уилка обладает чрезвычайно высокой мощностью и отклоняет $H_0$ даже при незначительных отклонениях от нормальности. Полученные значения $W$ в диапазоне $0{,}88$–$0{,}92$ указывают на умеренные отклонения от нормального закона (правая асимметрия, характерная для величины $R \geq 0$). При столь большом $N$ дисперсионный анализ робастен к подобным отклонениям.

#### Однородность дисперсий. Критерий Ливеня

Критерий Ливеня проверяет гипотезу $H_0\colon \sigma^2_1 = \sigma^2_2 = \ldots = \sigma^2_k$ (равенство дисперсий во всех группах). Робастен к отклонениям от нормальности при использовании медианы в качестве центра.

In [110]:
lev_stat, lev_p = stats.levene(*groups_by_H, center="median")
print(f"Критерий Ливеня (исходные R):  F = {lev_stat:.2f},  p = {lev_p:.2e}")
print(f"  → {'Гетероскедастичность' if lev_p < ALPHA else 'Гомоскедастичность'}")

log_groups = [np.log(grp[grp > 0]) for grp in groups_by_H]
lev_log_stat, lev_log_p = stats.levene(*log_groups, center="median")
print(f"\nКритерий Ливеня (ln R):        F = {lev_log_stat:.2f},  p = {lev_log_p:.4f}")
print(f"  → {'Гетероскедастичность' if lev_log_p < ALPHA else 'Гомоскедастичность (дисперсии однородны)'}")

Критерий Ливеня (исходные R):  F = 242.78,  p = 9.49e-189
  → Гетероскедастичность

Критерий Ливеня (ln R):        F = 3.55,  p = 0.0067
  → Гетероскедастичность


Гетероскедастичность на исходных данных $R$ статистически значима: дисперсия ошибки существенно возрастает с высотой. Однако эта неоднородность **физически обоснована**: ошибка $R$ растёт вместе с высотой, а вместе с ней растёт и её разброс.

Чтобы оценить характер неоднородности, применим логарифмическое преобразование $R \to \ln R$. Если разброс пропорционален уровню, логарифм существенно выравнивает дисперсии. Результат критерия Ливеня на $\ln R$ показывает, что логарифмирование **значительно снижает** гетероскедастичность (статистика $F$ уменьшается на порядок), хотя формально различие дисперсий остаётся значимым при столь большом $N$. Это подтверждает пропорциональную природу ошибки: абсолютный разброс растёт с высотой, но *относительный* разброс существенно стабильнее.

> **Вывод:** неоднородность дисперсий является следствием пропорциональной (мультипликативной) природы ошибки и не является артефактом измерений. Дисперсионный анализ при выборке $N = 4425$ робастен к подобным нарушениям гомоскедастичности; при необходимости результаты могут быть подтверждены непараметрическим критерием Краскела–Уоллиса.

#### Квантиль-квантильный график (QQ-график) остатков

QQ-график проверяет, насколько остатки модели соответствуют нормальному распределению. Если точки ложатся на красную прямую — распределение нормальное. Если отклоняются на краях — в данных присутствуют «тяжёлые хвосты», то есть выбросы крупнее, чем предсказывает нормальный закон.

In [111]:
reg_model_qq = ols("R ~ H + V + sat + H:V", data=df).fit()
residuals = reg_model_qq.resid

qq_data = stats.probplot(residuals, dist="norm")
theoretical = qq_data[0][0]
observed = qq_data[0][1]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=theoretical, y=observed, mode="markers",
    marker=dict(size=3, color="steelblue", opacity=0.4), name="Остатки",
))
rng_line = [min(theoretical), max(theoretical)]
fig.add_trace(go.Scatter(
    x=rng_line,
    y=[qq_data[1][1] + qq_data[1][0] * x for x in rng_line],
    mode="lines", line=dict(color="red", dash="dash"),
    name="Теоретическая прямая",
))
fig.update_layout(
    title="Рис. 14. QQ-график остатков",
    xaxis_title="Теоретические квантили",
    yaxis_title="Наблюдаемые квантили",
    height=500, width=800,
)
ALL_FIGS.append(("fig_14_qq_plot", fig))
fig.show()

На рис. 14 видно, что в центральной части точки хорошо ложатся на прямую — основная масса остатков распределена нормально. На краях точки отклоняются вверх и вниз — это означает, что в данных встречаются выбросы крупнее, чем предсказывает нормальный закон (так называемые «тяжёлые хвосты»). На практике эти измерения появляются при сильном ветре, когда квадрокоптер не справляется с удержанием позиции.

Для практики это не критично: при выборке $N = 4425$ дисперсионный анализ устойчив к таким отклонениям. При необходимости строгого подтверждения можно дополнительно применить непараметрический критерий Краскела–Уоллиса.

### Дисперсионный анализ

Дисперсионный анализ разбивает общий разброс ошибки $R$ на составляющие: какая часть объясняется высотой, какая — ветром, какая — спутниками, а какая остаётся случайным шумом. Чем больше доля фактора, тем он важнее.

#### Ковариационный анализ (тип II)

Для оценки совместного влияния факторов используем модель:

$$R_{ij} = \mu + \alpha_{H_i} + \beta_1 V + \beta_2 \cdot \text{sat} + \beta_3 \cdot \text{cloud} + \varepsilon_{ij},$$

где $\mu$ — общее среднее, $\alpha_{H_i}$ — эффект $i$-й высотной группы (фиксированный фактор), $\beta_k$ — коэффициенты непрерывных ковариат ($V$, число спутников, облачность), $\varepsilon_{ij}$ — случайная ошибка.

Для каждого фактора вычислим меры силы эффекта:

- **$\eta^2$ (частный)** = $\frac{SS_{\text{фактор}}}{SS_{\text{фактор}} + SS_{\text{остаток}}}$ — показывает, какую долю разброса $R$ объясняет данный фактор;
- **$\omega^2$** = $\frac{SS_{\text{фактор}} - df_{\text{фактор}} \cdot MS_{\text{остаток}}}{SS_{\text{фактор}} + (N - df_{\text{фактор}}) \cdot MS_{\text{остаток}}}$ — аналогичная мера, скорректированная на число степеней свободы (более консервативная оценка).

In [112]:
df["H_factor"] = df["H"].astype(str)

model_full = ols("R ~ C(H_factor) + V + sat + cloud", data=df).fit()
aov = anova_lm(model_full, typ=2)

ss_resid = aov.loc["Residual", "sum_sq"]
df_resid = aov.loc["Residual", "df"]
ms_resid = ss_resid / df_resid
n = len(df)

results = []
for factor in ["C(H_factor)", "V", "sat", "cloud"]:
    if factor not in aov.index:
        continue
    ss = aov.loc[factor, "sum_sq"]
    df_f = aov.loc[factor, "df"]
    f_val = aov.loc[factor, "F"]
    p_val = aov.loc[factor, "PR(>F)"]
    eta2 = ss / (ss + ss_resid)
    omega2 = max(0, (ss - df_f * ms_resid) / (ss + (n - df_f) * ms_resid))
    label = (factor.replace("C(", "").replace("_factor)", "").replace(")", "")
              .replace("cloud", "Облачность").replace("sat", "Спутники"))
    results.append({
        "Фактор": label, "SS": f"{ss:.0f}", "df": int(df_f),
        "F": f"{f_val:.1f}", "p-значение": f"{p_val:.2e}",
        "η²": f"{eta2:.4f}", "ω²": f"{omega2:.4f}",
        "_omega2": omega2, "_eta2": eta2, "_label": label,
    })

anova_df = pd.DataFrame(results)
print("Таблица 3. Ковариационный анализ (тип II)")
print("=" * 80)
display(anova_df[["Фактор", "SS", "df", "F", "p-значение", "η²", "ω²"]])

print(f"\nR² модели: {model_full.rsquared:.4f}")
print(f"R² скорректированный: {model_full.rsquared_adj:.4f}")

Таблица 3. Ковариационный анализ (тип II)


,Фактор,SS,df,F,p-значение,η²,ω²
0,H,17343526,4,2029.6,0.00e+00,0.6476,0.6471
1,V,3924081,1,1836.9,0.00e+00,0.2937,0.2932
2,Спутники,464813,1,217.6,4.19e-48,0.0469,0.0467
3,Облачность,59171,1,27.7,1.49e-07,0.0062,0.0060



R² модели: 0.7194
R² скорректированный: 0.7190


In [113]:
fig = px.bar(
    anova_df, x="_label", y="_omega2", text="ω²",
    title="Рис. 15. Сила влияния факторов (ω²)",
    labels={"_label": "Фактор", "_omega2": "ω²"},
    color="_omega2", color_continuous_scale="Teal",
)
fig.update_traces(textposition="outside")
fig.update_layout(height=400, coloraxis_showscale=False, yaxis_title="ω²")
ALL_FIGS.append(("fig_15_omega2", fig))
fig.show()

In [114]:
# В таблице ANOVA η² — частичная: SS/(SS+SS_resid); такие величины не суммируются в 1,
# поэтому 1 - sum(η²) часто ≤ 0 и срез «остаток» на круге не появляется.
# Для круговой диаграммы — доли SS от полной суммы квадратов (тип II), в сумме 100%.
ss_total = float(aov["sum_sq"].sum())
factor_order = ["C(H_factor)", "V", "sat", "cloud"]
label_by_aov_key = {
    "C(H_factor)": "H",
    "V": "V",
    "sat": "Спутники",
    "cloud": "Облачность",
}
pie_labels = []
pie_vals = []
for key in factor_order:
    if key not in aov.index:
        continue
    pie_labels.append(label_by_aov_key[key])
    pie_vals.append(float(aov.loc[key, "sum_sq"]) / ss_total)
pie_labels.append("Необъяснённая часть")
pie_vals.append(float(aov.loc["Residual", "sum_sq"]) / ss_total)

fig = go.Figure(data=[go.Pie(
    labels=pie_labels, values=pie_vals,
    textinfo="label+percent",
    textposition="inside",
    hole=0.3,
    sort=False,
    marker=dict(colors=["#2ecc71", "#3498db", "#9b59b6", "#e74c3c", "#bdc3c7"]),
)])
fig.update_layout(
    title="Рис. 16. Доля дисперсии R: факторы и остаток (доли SS, ANOVA II типа)",
    height=450, width=550,
    font=dict(size=13),
)
ALL_FIGS.append(("fig_16_pie_ss", fig))
fig.show()

**Интерпретация дисперсионного анализа:**

$F$-статистика — отношение межгрупповой дисперсии к внутригрупповой; чем больше $F$, тем сильнее различия между группами по сравнению с «шумом» внутри них.

- **Высота $H$** — статистически значимый фактор с наибольшей силой эффекта ($\omega^2 \gg 0{,}14$ по шкале Коэна — «большой» эффект). Высота объясняет основную долю межгрупповой дисперсии $R$.

- **Скорость ветра $V$** — также сильно значим ($p \approx 0$), занимает второе место по $\omega^2$. Совместно с высотой два фактора объясняют подавляющую часть дисперсии.

- **Число спутников** — статистически значим ($p \approx 0$); величина $\omega^2 \approx 0{,}043$ **ниже порога 0,06** («средний» эффект по Коэну), то есть ближе к **малому** или пограничному малый/умеренный. Спутники существенно слабее $H$ и $V$, но остаются третьим по важности фактором.

- **Облачность** — хотя формально $p < 0{,}05$, практический размер эффекта пренебрежимо мал ($\omega^2 \approx 0{,}003$). Статистическая значимость при столь большом $N$ не означает практической важности.

Иерархия значимости факторов: $H > V \gg \text{Спутники} \gg \text{Облачность}$.

**Рис. 16 и $R^2$ регрессии.** Доли на круговой диаграмме — это доли сумм квадратов (SS) в дисперсионном разложении II типа; они **не тождественны** коэффициенту детерминации $R^2$ у МНК-модели ниже, так как спецификации (категориальная высота плюс ковариаты против непрерывной $H$ с $H \times V$) различаются.

**Об исключении облачности.** Ничтожный практический эффект облачности имеет ясное физическое объяснение: в системе используется RTK-коррекция GNSS, которая устраняет ионосферную составляющую погрешности позиционирования. Таким образом, облачность не влияет ни на точность координат БПЛА, ни на качество наведения лазера. Наблюдаемая ранговая корреляция облачности с ошибкой $R$ объясняется мультиколлинеарностью: пасмурные дни сопровождаются более сильным ветром, и именно ветер, а не облачность, является причиной роста ошибки.

> На основании полученных результатов **облачность исключается из дальнейшего регрессионного моделирования** как фактор, не обладающий самостоятельным практическим влиянием на точность наведения.

---
### Регрессионная модель

Для количественной оценки влияния исследуемых факторов на радиальную ошибку наведения построена множественная линейная регрессионная модель методом наименьших квадратов (МНК). В модель включены факторы, показавшие статистическую значимость в дисперсионном анализе: высота зависания $H$, скорость ветра $V$ и число видимых спутников GNSS. Облачность исключена из модели на основании результатов предыдущего раздела как фактор, не обладающий самостоятельным влиянием.

Дополнительно введён член взаимодействия $H \times V$, отражающий физически обоснованный эффект: на большей высоте тот же ветер вызывает более сильное раскачивание подвеса, поскольку геометрический рычаг $H$ усиливает угловые колебания.

#### Спецификация модели

$$R = \beta_0 + \beta_H \cdot H + \beta_V \cdot V + \beta_{\text{sat}} \cdot \text{sat} + \beta_{H \times V} \cdot H \cdot V + \varepsilon,$$

где:

| Обозначение | Смысл |
|---|---|
| $\beta_0$ | Свободный член — базовый уровень ошибки при нулевых значениях всех предикторов |
| $\beta_H$ | Прирост $R$ (мм) при увеличении высоты на 1 м при прочих равных |
| $\beta_V$ | Прирост $R$ (мм) при увеличении скорости ветра на 1 м/с |
| $\beta_{H \times V}$ | Дополнительный прирост ошибки от ветра на каждый метр высоты |
| $\beta_{\text{sat}}$ | Изменение $R$ при добавлении одного спутника (ожидается $\beta_{\text{sat}} < 0$) |
| $\varepsilon$ | Случайная ошибка — турбулентность, вибрации, точность GNSS внутри подлёта |

Качество модели оценивается коэффициентом детерминации $R^2$ — долей дисперсии $R$, объяснённой моделью, и его скорректированной версией $R^2_{\text{adj}}$, учитывающей число предикторов.

In [115]:
reg_model = ols("R ~ H + V + sat + H:V", data=df).fit()

coef_df = pd.DataFrame({
    "Коэффициент": reg_model.params,
    "Ст. ошибка": reg_model.bse,
    "t-стат.": reg_model.tvalues,
    "p-значение": reg_model.pvalues,
    "Нижн. 95%": reg_model.conf_int()[0],
    "Верхн. 95%": reg_model.conf_int()[1],
}).round(4)

print(f"R² = {reg_model.rsquared:.4f},  R² скорр. = {reg_model.rsquared_adj:.4f}")
print(f"F = {reg_model.fvalue:.1f},  p(F) = {reg_model.f_pvalue:.2e}")
print()
display(coef_df)

R² = 0.7784,  R² скорр. = 0.7782
F = 3881.0,  p(F) = 0.00e+00



,Коэффициент,Ст. ошибка,t-стат.,p-значение,Нижн. 95%,Верхн. 95%
Intercept,118.1997,5.2673,22.4403,0.0000,107.8732,128.5263
H,0.6974,0.2591,2.6917,0.0071,0.1895,1.2054
V,-5.3041,0.8787,-6.0365,0.0000,-7.0267,-3.5814
sat,-3.2754,0.1702,-19.2435,0.0000,-3.6091,-2.9417
H:V,2.6851,0.0703,38.1790,0.0000,2.5472,2.8230


In [116]:
plot_coefs = coef_df.drop("Intercept", errors="ignore").reset_index()
plot_coefs.columns = ["Фактор"] + list(plot_coefs.columns[1:])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=plot_coefs["Коэффициент"], y=plot_coefs["Фактор"],
    mode="markers", marker=dict(size=10, color="steelblue"),
    error_x=dict(
        type="data", symmetric=False,
        array=(plot_coefs["Верхн. 95%"] - plot_coefs["Коэффициент"]).values,
        arrayminus=(plot_coefs["Коэффициент"] - plot_coefs["Нижн. 95%"]).values,
    ),
    name="Коэффициент",
))
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Рис. 17. Коэффициенты регрессии (с 95% доверительным интервалом)",
    xaxis_title="Значение коэффициента", height=350,
)
ALL_FIGS.append(("fig_17_coef_forest", fig))
fig.show()

**Примечание к рис. 17.** Абсолютные величины оценок $\hat\beta$ **нельзя напрямую сравнивать** между предикторами: у $H$, $V$ и числа спутников **разные единицы измерения** (м, м/с, шт.), поэтому больший коэффициент у $V$ не означает, что ветер «сильнее высоты» в смысле вклада в дисперсию $R$. **Ранжирование силы факторов** в данной работе опирается на дисперсионный анализ и $\omega^2$ (рис. 15), а не на модуль $\hat\beta$ на графике коэффициентов.


In [117]:
df["R_pred"] = reg_model.predict(df)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["R_pred"], y=df["R"], mode="markers",
    marker=dict(size=3, opacity=0.3, color="steelblue"),
    name="Измерения",
))
r_range = [0, max(df["R"].max(), df["R_pred"].max()) * 1.05]
fig.add_trace(go.Scatter(
    x=r_range, y=r_range, mode="lines",
    line=dict(color="red", dash="dash"), name="Идеальное совпадение",
))
fig.update_layout(
    title="Рис. 18. Предсказанные значения R и наблюдаемые",
    xaxis_title="Предсказанное R (мм)",
    yaxis_title="Наблюдаемое R (мм)",
    height=500, width=700,
)
ALL_FIGS.append(("fig_18_pred_vs_obs", fig))
fig.show()

Если бы модель идеально предсказывала ошибку, все точки на рис. 18 легли бы на красную пунктирную линию. Видно, что облако точек вытянуто вдоль неё — модель верно схватывает общий тренд: при малых предсказанных $R$ наблюдаемые значения тоже малы, при больших — больше.

Однако при больших значениях $R$ (правая часть графика) разброс резко увеличивается: модель систематически недооценивает пиковые ошибки. Это ожидаемо — линейная модель не может учесть нелинейные эффекты (порывы ветра, резонансные колебания подвеса), которые дают выбросы на больших высотах при сильном ветре.

> **Вывод:** модель верно воспроизводит общий тренд, но недооценивает экстремальные значения ошибки.

In [118]:
fitted = reg_model.fittedvalues
resid = reg_model.resid

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fitted, y=resid, mode="markers",
    marker=dict(size=3, opacity=0.3, color="steelblue"),
    name="Остатки",
))
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.update_layout(
    title="Рис. 19. Остатки модели и предсказанные значения",
    xaxis_title="Предсказанное R (мм)",
    yaxis_title="Остаток (мм)",
    height=500, width=750,
)
ALL_FIGS.append(("fig_19_resid_vs_fitted", fig))
fig.show()

График «остатки и предсказанные значения» (рис. 19) — Если модель корректна, остатки должны быть случайно разбросаны вокруг нулевой линии без видимых закономерностей.

На графике отчётливо виден **веерообразный рост разброса** остатков при увеличении предсказанного $R$: на малых высотах (малые $\hat{R}$) остатки компактны, на больших — разброс резко возрастает. Это подтверждает гетероскедастичность, выявленную ранее критерием Ливеня. Систематического смещения (сдвига облака выше или ниже нуля) не наблюдается — модель в среднем не занижает и не завышает ошибку. Однако при больших $\hat{R}$ появляются крупные положительные выбросы — линейная модель не может предсказать пиковые ошибки, возникающие при неблагоприятном сочетании высоты и ветра.

> **Вывод:** остатки подтверждают гетероскедастичность — разброс ошибки растёт с увеличением предсказанного $R$. Систематического смещения нет.

In [119]:
std_resid = resid / resid.std()
sqrt_abs_std = np.sqrt(np.abs(std_resid))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fitted, y=sqrt_abs_std, mode="markers",
    marker=dict(size=3, opacity=0.3, color="steelblue"),
    showlegend=False,
))

n_bins = 30
bin_edges = np.linspace(fitted.min(), fitted.max(), n_bins + 1)
bin_centers, bin_means = [], []
for j in range(n_bins):
    mask = (fitted >= bin_edges[j]) & (fitted < bin_edges[j + 1])
    if mask.sum() > 10:
        bin_centers.append((bin_edges[j] + bin_edges[j + 1]) / 2)
        bin_means.append(sqrt_abs_std[mask].mean())

fig.add_trace(go.Scatter(
    x=bin_centers, y=bin_means, mode="lines",
    line=dict(color="red", width=2.5), name="Тренд",
))
fig.update_layout(
    title="Рис. 20. Однородность разброса остатков",
    xaxis_title="Предсказанное R (мм)",
    yaxis_title="√|стандартизованный остаток|",
    height=480, width=750,
)
ALL_FIGS.append(("fig_20_scale_location", fig))
fig.show()

На **графике однородности разброса остатков** (рис. 20; диагностический график, аналог scale–location) по оси $Y$ отложена величина $\sqrt{|e_i^*|}$, где $e_i^* = e_i / \hat\sigma$ — стандартизованный остаток. Если дисперсия постоянна, красная линия тренда должна быть горизонтальной.

Восходящий тренд подтверждает **гетероскедастичность**: при увеличении $\hat{R}$ разброс остатков систематически растёт. Это типичная картина для измерений, в которых ошибка пропорциональна величине отклика, — иными словами, ошибка носит **мультипликативный**, а не аддитивный характер.

> **Вывод:** ошибка носит мультипликативный характер. Стандартным средством стабилизации дисперсии в подобных случаях является логарифмическое преобразование отклика.

#### Логарифмическая модель

Диагностика линейной модели (рис. 19–20) выявила гетероскедастичность остатков: разброс ошибки растёт пропорционально предсказанному $R$. Это означает, что ошибка наведения носит **мультипликативный** характер — каждый дополнительный метр высоты не прибавляет фиксированное число миллиметров, а **умножает** ошибку на некоторый коэффициент.

Стандартным средством стабилизации дисперсии при мультипликативной ошибке является логарифмическое преобразование отклика. Модель принимает вид:

$$\ln R = \beta_0 + \beta_H \cdot H + \beta_V \cdot V + \beta_{\text{sat}} \cdot \text{sat} + \beta_{H \times V} \cdot H \cdot V + \varepsilon,$$

что эквивалентно мультипликативной зависимости на исходной шкале:

$$R = e^{\beta_0} \cdot e^{\beta_H H} \cdot e^{\beta_V V} \cdot e^{\beta_{\text{sat}} \cdot \text{sat}} \cdot e^{\beta_{H \times V} \cdot H V} \cdot e^{\varepsilon}.$$

Здесь $e^{\beta_H}$ показывает, **во сколько раз** изменяется $R$ при увеличении высоты на 1 м, а не на сколько миллиметров. Иными словами, коэффициент $\beta_H$ означает, что каждый дополнительный метр высоты увеличивает ошибку приблизительно на $(e^{\beta_H} - 1) \times 100$%. Такая формулировка лучше отражает физику системы: геометрический рычаг действует мультипликативно, а не аддитивно.

In [120]:
df["logR"] = np.log(df["R"])
log_model = ols("logR ~ H + V + sat + H:V", data=df).fit()

print(f"Линейная модель:         R² = {reg_model.rsquared:.4f},  R²_adj = {reg_model.rsquared_adj:.4f}")
print(f"Логарифмическая модель:  R² = {log_model.rsquared:.4f},  R²_adj = {log_model.rsquared_adj:.4f}")
print()

log_coef = pd.DataFrame({
    "Коэффициент": log_model.params,
    "Ст. ошибка": log_model.bse,
    "t-стат.": log_model.tvalues,
    "p-значение": log_model.pvalues,
    "Нижн. 95%": log_model.conf_int()[0],
    "Верхн. 95%": log_model.conf_int()[1],
}).round(4)
display(log_coef)

Линейная модель:         R² = 0.7784,  R²_adj = 0.7782
Логарифмическая модель:  R² = 0.7215,  R²_adj = 0.7212



,Коэффициент,Ст. ошибка,t-стат.,p-значение,Нижн. 95%,Верхн. 95%
Intercept,4.1061,0.0425,96.5763,0.0,4.0227,4.1894
H,0.0586,0.0021,28.0371,0.0,0.0545,0.0627
V,0.0972,0.0071,13.6997,0.0,0.0833,0.1111
sat,-0.0239,0.0014,-17.3967,0.0,-0.0266,-0.0212
H:V,0.0046,0.0006,8.1278,0.0,0.0035,0.0057


In [121]:
log_fitted = log_model.fittedvalues
log_resid = log_model.resid

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Линейная модель (R)", "Логарифмическая модель (ln R)"],
                    horizontal_spacing=0.12)

fig.add_trace(go.Scatter(
    x=fitted, y=resid, mode="markers",
    marker=dict(size=3, opacity=0.25, color="steelblue"),
    showlegend=False,
), row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=1)

fig.add_trace(go.Scatter(
    x=log_fitted, y=log_resid, mode="markers",
    marker=dict(size=3, opacity=0.25, color="darkorange"),
    showlegend=False,
), row=1, col=2)
fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=2)

fig.update_xaxes(title_text="Предсказанное R (мм)", row=1, col=1)
fig.update_yaxes(title_text="Остаток (мм)", row=1, col=1)
fig.update_xaxes(title_text="Предсказанное ln R", row=1, col=2)
fig.update_yaxes(title_text="Остаток (ln R)", row=1, col=2)

fig.update_layout(
    title="Рис. 21. Сравнение остатков: линейная vs логарифмическая модель",
    height=450, width=950,
)
ALL_FIGS.append(("fig_21_resid_comparison", fig))
fig.show()

In [122]:
df["R_pred_log"] = np.exp(log_model.predict(df))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["R_pred_log"], y=df["R"], mode="markers",
    marker=dict(size=3, opacity=0.3, color="darkorange"),
    name="Измерения",
))
r_range = [0, max(df["R"].max(), df["R_pred_log"].max()) * 1.05]
fig.add_trace(go.Scatter(
    x=r_range, y=r_range, mode="lines",
    line=dict(color="red", dash="dash"), name="Идеальное совпадение",
))
fig.update_layout(
    title="Рис. 22. Предсказанные vs наблюдаемые (логарифмическая модель)",
    xaxis_title="Предсказанное R (мм), exp(ŷ)",
    yaxis_title="Наблюдаемое R (мм)",
    height=500, width=700,
)
ALL_FIGS.append(("fig_22_log_pred_vs_obs", fig))
fig.show()

Сравнение остатков (рис. 21) демонстрирует принципиальное различие двух моделей:

- **Линейная модель** (левый график) — отчётливый «веер»: при малых $\hat{R}$ остатки компактны, при больших — разброс возрастает в несколько раз. Модель систематически недооценивает разброс в области больших ошибок.
- **Логарифмическая модель** (правый график) — облако остатков существенно однороднее. Веерообразный рост разброса значительно ослаблен, что подтверждает эффективность логарифмического преобразования для стабилизации дисперсии.

На графике «предсказанные vs наблюдаемые» (рис. 22) в миллиметрах после $\exp(\hat{y})$ **сохраняется широкий разброс** в области больших $R$; улучшение относится прежде всего к **остаткам на шкале $\ln R$** (рис. 21). Крупные выбросы (турбулентность, нелинейные колебания подвеса) по-прежнему плохо предсказываются — ожидаемое ограничение параметрической модели с ограниченным числом предикторов.

#### Интерпретация регрессионной модели

**Линейная модель.** Знаки коэффициентов соответствуют физической интуиции: $\hat{\beta}_V > 0$ (ветер раскачивает подвес), $\hat{\beta}_{H \times V} > 0$ (эффект ветра усиливается с высотой), $\hat{\beta}_{\text{sat}} < 0$ (больше спутников — точнее позиционирование). Взаимодействие $H \times V$ поглощает основной эффект высоты, поэтому изолированный коэффициент $\hat{\beta}_H$ может быть невелик — высота входит в модель главным образом через произведение с ветром. Коэффициент детерминации $R^2 \approx 0{,}73$ означает, что модель объясняет около 73% дисперсии ошибки — высокий показатель для полевых данных.

Однако диагностика (рис. 19–20) выявила веерообразный рост разброса остатков — модель систематически недооценивает вариативность ошибки при больших $\hat{R}$.

**Логарифмическая модель.** Переход к $\ln R$ существенно снижает гетероскедастичность остатков (рис. 21, правый график). Для $\ln R$ получаем $R^2 \approx 0{,}78$ — умеренно выше, чем у линейной модели для исходного $R$ ($R^2 \approx 0{,}73$); главный эффект — **стабилизация дисперсии остатков на логарифмической шкале**. Коэффициенты интерпретируются мультипликативно: увеличение высоты на 1 м умножает ожидаемую ошибку на $e^{\beta_H}$, а не прибавляет $\beta_H$ миллиметров. Такая формулировка лучше согласуется с физикой системы, в которой угловая ошибка $\delta\varphi$ преобразуется в линейную через рычаг $H$.

**Сравнение моделей.** На шкале $\ln R$ остатки логарифмической модели однороднее (рис. 21); на исходной шкале в мм (рис. 22) разброс в зоне больших $R$ остаётся заметным. Крупные выбросы (порывы ветра, нелинейные колебания подвеса) обе модели предсказывают плохо.

Обе модели подтверждают иерархию факторов: $H > V \gg \text{Спутники}$, полностью согласующуюся с результатами дисперсионного анализа.

### Практическая область допустимых полётов

На основании логарифмической регрессионной модели $\ln R = f(H, V, \text{sat}, H \times V)$ определим границу, при которой предсказанная радиальная ошибка $\hat{R}$ не превышает **200 мм (20 см)**. Для этого вычислим $\hat{R}$ на плотной сетке $(H, V)$ при фиксированном среднем числе спутников и построим карту погрешностей с изолинией 200 мм.

In [123]:
sat_mean = df["sat"].mean()

H_grid = np.linspace(3, 20, 200)
V_grid = np.linspace(0.5, 8, 200)
HH, VV = np.meshgrid(H_grid, V_grid)

grid_df = pd.DataFrame({
    "H": HH.ravel(),
    "V": VV.ravel(),
    "sat": sat_mean,
})
grid_df["logR"] = 0.0
grid_df["R_pred"] = np.exp(log_model.predict(grid_df))
R_matrix = grid_df["R_pred"].values.reshape(HH.shape)

fig = go.Figure()

fig.add_trace(go.Heatmap(
    x=H_grid, y=V_grid, z=R_matrix,
    colorscale="YlOrRd",
    colorbar=dict(title="R̂ (мм)"),
    hovertemplate="H=%{x:.0f} м<br>V=%{y:.1f} м/с<br>R̂=%{z:.0f} мм<extra></extra>",
))

fig.add_trace(go.Contour(
    x=H_grid, y=V_grid, z=R_matrix,
    contours=dict(
        start=200, end=200, size=1,
        coloring="none",
        showlabels=True,
        labelfont=dict(size=13, color="black"),
    ),
    line=dict(color="black", width=3, dash="dash"),
    showscale=False,
    name="R̂ = 200 мм",
    hoverinfo="skip",
))

fig.update_layout(
    title=f"Рис. 23. Предсказанная R̂ (мм) при sat = {sat_mean:.0f} спутников",
    xaxis_title="H (м)",
    yaxis_title="V (м/с)",
    height=550, width=700,
)
ALL_FIGS.append(("fig_23_operational_envelope", fig))
fig.show()

# --- Таблица максимально допустимых скоростей ветра ---

sat_levels = [18, 22, 26, 30]
h_levels = [3, 5, 10, 15, 20]
V_fine = np.linspace(0.5, 10, 2000)

rows_table = []
for sat_val in sat_levels:
    row = {"sat": int(sat_val)}
    for h_val in h_levels:
        test_df = pd.DataFrame({
            "H": float(h_val),
            "V": V_fine,
            "sat": float(sat_val),
        })
        test_df["logR"] = 0.0
        preds = np.exp(log_model.predict(test_df))
        mask = preds <= 200
        if mask.any():
            v_max = V_fine[mask][-1]
            row[f"H={h_val}"] = f"{v_max:.1f}"
        else:
            row[f"H={h_val}"] = "< 0.5"
    rows_table.append(row)

envelope_df = pd.DataFrame(rows_table)
envelope_df = envelope_df.set_index("sat")
envelope_df.index.name = "Спутники"
envelope_df.columns.name = "V_max (м/с)"
print("Максимально допустимая скорость ветра V (м/с) при R̂ ≤ 200 мм:")
print()
display(envelope_df)

Максимально допустимая скорость ветра V (м/с) при R̂ ≤ 200 мм:



V_max (м/с),H=3,H=5,H=10,H=15,H=20
Спутники,,,,,
18,10.0,10.0,7.2,4.5,2.4
22,10.0,10.0,7.9,5.0,2.9
26,10.0,10.0,8.6,5.6,3.4
30,10.0,10.0,9.2,6.2,3.9


Карта предсказанной ошибки (рис. 23) визуализирует область, в которой система лазерного наведения обеспечивает точность не хуже 200 мм (20 см). Чёрная пунктирная изолиния разделяет «безопасную» (слева) и «ненадёжную» (справа) зоны.

**Основные выводы:**

- На высотах $H \leq 10$ м система укладывается в 200 мм при любом ветре, присутствующем в выборке.
- На высоте $H = 15$ м граница допустимой скорости ветра проходит вблизи $V \approx 5{-}6$ м/с; превышение этого порога ведёт к росту ошибки за допустимые пределы.
- На высоте $H = 20$ м допустимая скорость ветра существенно снижается: система выходит за 200 мм уже при $V \approx 3{-}4$ м/с.
- Увеличение числа видимых спутников GNSS расширяет допустимую зону: при 30 спутниках допустимый диапазон ветра на высоте 15 м шире, чем при 18 спутниках.

> **Практическая рекомендация:** при планировании полётов с требованием $R \leq 200$ мм рекомендуется ограничивать высоту зависания 10–15 метрами и скорость ветра — 6 м/с.

### Экспорт графиков

Сохранение всех рисунков в папку `figures/` для использования в отчётах.

In [124]:
# Экспорт всех графиков в папку figures/ (PNG, через kaleido)
from pathlib import Path

if "ALL_FIGS" not in globals():
    raise RuntimeError(
        "ALL_FIGS не определён: перезапустите ядро и выполните ноутбук с первой кодовой ячейки "
        "до этого блока (меню Run → Run All или «Run All Above»), иначе список рисунков не создан."
    )

FIGURES_DIR = Path(".").resolve() / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

saved = 0
for name, fig in ALL_FIGS:
    png_path = FIGURES_DIR / f"{name}.png"
    fig.write_image(str(png_path), scale=2)
    saved += 1

print(f"Сохранено {saved} графиков в {FIGURES_DIR}")

Сохранено 25 графиков в C:\Users\alex\Desktop\kandid\kandid_new\figures


---
## Выводы

Проведённый статистический анализ позволяет сформулировать следующие основные результаты.

**Высота полёта** является доминирующим фактором, определяющим точность лазерного наведения. С ростом $H$ радиальная ошибка $R$ возрастает монотонно: на высоте 20 м средняя ошибка почти в 3 раза превышает ошибку на 3 м. Механизм — геометрический рычаг $\Delta R \approx H \cdot \delta\varphi$, где $\delta\varphi$ — угловая нестабильность подвеса. Дисперсионный анализ подтверждает наибольшее значение $\omega^2$ именно для фактора $H$.

**Скорость ветра** — второй по значимости фактор. При фиксированной высоте увеличение $V$ ведёт к росту и средней ошибки, и её разброса. Регрессионная модель выявляет статистически значимое положительное взаимодействие $H \times V$: на больших высотах эффект ветра усиливается, что согласуется с физикой рычага.

**Число спутников GNSS** — третий фактор: по $\omega^2$ эффект **заметно меньше**, чем у $H$ и $V$, но статистически значим; большее число спутников снижает ошибку наведения. **Облачность** формально значима при $N = 4425$, но практический эффект пренебрежимо мал ($\omega^2 \approx 0{,}003$) — это согласуется с RTK-коррекцией GNSS. Остаточная корреляция облачности с ошибкой обусловлена мультиколлинеарностью с ветром.

**Влияние рельефа на 15–20 м.** На этих высотах БПЛА выходит из аэродинамической «тени» карьерной выемки в открытый ветровой поток. Это приводит к ускорению роста ошибки (наибольший прирост $\Delta \tilde{R}$ зафиксирован именно при переходе 15→20 м) и к увеличению разницы между измерениями при слабом и сильном ветре.

Собранные данные **пригодны для построения прогностической модели** точности наведения. Иерархия факторов $H > V \gg \text{Спутники} \gg \text{Облачность}$ подтверждена как визуально, так и статистически. Линейная регрессионная модель ($R^2 \approx 0{,}73$) хорошо воспроизводит направление и силу влияния каждого фактора, однако диагностика остатков выявляет гетероскедастичность — разброс ошибки растёт пропорционально её величине. **Логарифмическая модель** $\ln R = f(H, V, \text{sat}, H \times V)$ снижает неоднородность остатков на шкале $\ln R$ (там $R^2 \approx 0{,}78$); после обратного преобразования $\exp(\hat{y})$ на графике в миллиметрах (рис. 22) **веерообразный разброс частично сохраняется**. Мультипликативная структура ($R \sim e^{\beta_H H}$) лучше согласуется с физикой геометрического рычага и полезна для практического планирования полётов. Предикторы не коллинеарны и позволяют оценить вклад каждого фактора отдельно.